# MSSQL to Databricks Parallel Table Loader

## Overview
This notebook loads 100+ tables from MSSQL Server into Databricks Delta Lake using JDBC with:
- **Parallelism**: Concurrent table loading with partitioned reads for large tables
- **Cost Efficiency**: Optimized partitioning, caching strategies, and resource management
- **SOLID Principles**: Clean, maintainable, and extensible architecture

## Architecture
- **Single Responsibility**: Each class has one job (config, connection, loading, orchestration)
- **Open/Closed**: Base classes can be extended without modification
- **Liskov Substitution**: All loaders implement the same interface
- **Interface Segregation**: Separate interfaces for different concerns
- **Dependency Inversion**: High-level modules depend on abstractions

## Cell 1: Configuration & Widgets

In [ ]:
# Databricks notebook widgets for parameterization
dbutils.widgets.text("mssql_host", "your-mssql-server.database.windows.net", "MSSQL Host")
dbutils.widgets.text("mssql_port", "1433", "MSSQL Port")
dbutils.widgets.text("mssql_database", "your_database", "MSSQL Database")
dbutils.widgets.text("mssql_schema", "dbo", "MSSQL Schema")
dbutils.widgets.text("target_catalog", "your_catalog", "Target Catalog")
dbutils.widgets.text("target_schema", "bronze", "Target Schema")
dbutils.widgets.dropdown("load_mode", "overwrite", ["overwrite", "append", "merge"], "Load Mode")
dbutils.widgets.text("max_parallel_tables", "10", "Max Parallel Tables")
dbutils.widgets.text("partition_size_mb", "128", "Partition Size (MB)")

## Cell 2: Imports and Dependencies

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Protocol, Callable, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import logging
import json

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType
from delta.tables import DeltaTable

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Cell 3: Configuration Classes (Single Responsibility Principle)

In [ ]:
@dataclass(frozen=True)
class JDBCConfig:
    """Immutable JDBC connection configuration - Single Responsibility: holds connection details only."""
    host: str
    port: int
    database: str
    driver: str = "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    
    @property
    def jdbc_url(self) -> str:
        return f"jdbc:sqlserver://{self.host}:{self.port};databaseName={self.database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"


@dataclass(frozen=True)
class TableConfig:
    """Configuration for a single table load - Single Responsibility: holds table metadata only."""
    source_schema: str
    source_table: str
    target_catalog: str
    target_schema: str
    target_table: Optional[str] = None
    partition_column: Optional[str] = None
    lower_bound: Optional[int] = None
    upper_bound: Optional[int] = None
    num_partitions: Optional[int] = None
    custom_query: Optional[str] = None
    primary_keys: List[str] = field(default_factory=list)
    
    @property
    def full_source_name(self) -> str:
        return f"{self.source_schema}.{self.source_table}"
    
    @property
    def full_target_name(self) -> str:
        target = self.target_table or self.source_table
        return f"{self.target_catalog}.{self.target_schema}.{target}"


@dataclass
class LoaderConfig:
    """Overall loader configuration - Single Responsibility: orchestration settings only."""
    jdbc_config: JDBCConfig
    max_parallel_tables: int = 10
    partition_size_mb: int = 128
    fetch_size: int = 10000
    batch_size: int = 100000
    load_mode: str = "overwrite"  # overwrite, append, merge
    enable_push_down: bool = True
    
    def __post_init__(self):
        if self.max_parallel_tables < 1:
            raise ValueError("max_parallel_tables must be >= 1")
        if self.partition_size_mb < 1:
            raise ValueError("partition_size_mb must be >= 1")

## Cell 4: Abstract Interfaces (Interface Segregation & Dependency Inversion)

In [ ]:
class IConnectionProvider(Protocol):
    """Interface for providing database connections - Interface Segregation."""
    def get_connection_properties(self) -> Dict[str, str]: ...
    def get_jdbc_url(self) -> str: ...


class ITableMetadataProvider(Protocol):
    """Interface for providing table metadata - Interface Segregation."""
    def get_row_count(self, table_config: TableConfig) -> int: ...
    def get_partition_bounds(self, table_config: TableConfig) -> tuple: ...
    def get_primary_keys(self, table_config: TableConfig) -> List[str]: ...


class IDataReader(Protocol):
    """Interface for reading data - Interface Segregation."""
    def read(self, table_config: TableConfig) -> DataFrame: ...


class IDataWriter(Protocol):
    """Interface for writing data - Interface Segregation."""
    def write(self, df: DataFrame, table_config: TableConfig, mode: str) -> None: ...


class ITableLoader(Protocol):
    """Interface for complete table loading - combines read and write."""
    def load(self, table_config: TableConfig) -> Dict[str, Any]: ...

## Cell 5: Connection Provider (Single Responsibility)

In [ ]:
class MSSQLConnectionProvider:
    """Provides MSSQL connection details - Single Responsibility: connection management only."""
    
    def __init__(self, jdbc_config: JDBCConfig, secret_scope: str = "mssql-secrets"):
        self._jdbc_config = jdbc_config
        self._secret_scope = secret_scope
    
    def get_connection_properties(self) -> Dict[str, str]:
        """Get JDBC connection properties with credentials from secret scope."""
        return {
            "user": dbutils.secrets.get(scope=self._secret_scope, key="username"),
            "password": dbutils.secrets.get(scope=self._secret_scope, key="password"),
            "driver": self._jdbc_config.driver,
            "fetchsize": "10000",
            "batchsize": "100000",
            # Cost optimization: enable query pushdown
            "pushDownPredicate": "true",
            "pushDownAggregate": "true",
        }
    
    def get_jdbc_url(self) -> str:
        return self._jdbc_config.jdbc_url

## Cell 6: Table Metadata Provider (Single Responsibility)

In [ ]:
class MSSQLMetadataProvider:
    """Provides table metadata from MSSQL - Single Responsibility: metadata queries only."""
    
    def __init__(self, spark: SparkSession, connection_provider: IConnectionProvider):
        self._spark = spark
        self._conn_provider = connection_provider
    
    def get_row_count(self, table_config: TableConfig) -> int:
        """Get approximate row count for partitioning decisions."""
        query = f"""
            (SELECT SUM(p.rows) as row_count
             FROM sys.partitions p
             JOIN sys.tables t ON p.object_id = t.object_id
             JOIN sys.schemas s ON t.schema_id = s.schema_id
             WHERE s.name = '{table_config.source_schema}'
               AND t.name = '{table_config.source_table}'
               AND p.index_id IN (0, 1)) as row_count_query
        """
        df = self._spark.read.jdbc(
            url=self._conn_provider.get_jdbc_url(),
            table=query,
            properties=self._conn_provider.get_connection_properties()
        )
        result = df.collect()[0][0]
        return int(result) if result else 0
    
    def get_partition_bounds(self, table_config: TableConfig) -> tuple:
        """Get min/max values for partition column."""
        if not table_config.partition_column:
            return (None, None)
        
        query = f"""
            (SELECT MIN({table_config.partition_column}) as min_val,
                    MAX({table_config.partition_column}) as max_val
             FROM {table_config.full_source_name}) as bounds_query
        """
        df = self._spark.read.jdbc(
            url=self._conn_provider.get_jdbc_url(),
            table=query,
            properties=self._conn_provider.get_connection_properties()
        )
        row = df.collect()[0]
        return (row[0], row[1])
    
    def get_primary_keys(self, table_config: TableConfig) -> List[str]:
        """Get primary key columns for merge operations."""
        query = f"""
            (SELECT c.name as column_name
             FROM sys.indexes i
             JOIN sys.index_columns ic ON i.object_id = ic.object_id AND i.index_id = ic.index_id
             JOIN sys.columns c ON ic.object_id = c.object_id AND ic.column_id = c.column_id
             JOIN sys.tables t ON i.object_id = t.object_id
             JOIN sys.schemas s ON t.schema_id = s.schema_id
             WHERE i.is_primary_key = 1
               AND s.name = '{table_config.source_schema}'
               AND t.name = '{table_config.source_table}') as pk_query
        """
        df = self._spark.read.jdbc(
            url=self._conn_provider.get_jdbc_url(),
            table=query,
            properties=self._conn_provider.get_connection_properties()
        )
        return [row[0] for row in df.collect()]
    
    def get_numeric_columns(self, table_config: TableConfig) -> List[str]:
        """Get numeric columns suitable for partitioning."""
        query = f"""
            (SELECT c.name
             FROM sys.columns c
             JOIN sys.types t ON c.user_type_id = t.user_type_id
             JOIN sys.tables tb ON c.object_id = tb.object_id
             JOIN sys.schemas s ON tb.schema_id = s.schema_id
             WHERE s.name = '{table_config.source_schema}'
               AND tb.name = '{table_config.source_table}'
               AND t.name IN ('int', 'bigint', 'smallint', 'tinyint')
             ORDER BY c.column_id) as numeric_cols_query
        """
        df = self._spark.read.jdbc(
            url=self._conn_provider.get_jdbc_url(),
            table=query,
            properties=self._conn_provider.get_connection_properties()
        )
        return [row[0] for row in df.collect()]

## Cell 7: Partition Strategy (Open/Closed Principle)

In [ ]:
class PartitionStrategy(ABC):
    """Abstract base for partition strategies - Open/Closed: extend without modifying."""
    
    @abstractmethod
    def calculate_partitions(self, table_config: TableConfig, row_count: int) -> Dict[str, Any]:
        """Calculate optimal partition parameters."""
        pass


class AdaptivePartitionStrategy(PartitionStrategy):
    """Adaptive partitioning based on table size - Cost efficient."""
    
    def __init__(self, 
                 metadata_provider: ITableMetadataProvider,
                 partition_size_mb: int = 128,
                 avg_row_size_bytes: int = 500):
        self._metadata_provider = metadata_provider
        self._partition_size_mb = partition_size_mb
        self._avg_row_size_bytes = avg_row_size_bytes
    
    def calculate_partitions(self, table_config: TableConfig, row_count: int) -> Dict[str, Any]:
        """Calculate optimal partitions based on data size."""
        # Estimate table size in MB
        estimated_size_mb = (row_count * self._avg_row_size_bytes) / (1024 * 1024)
        
        # For small tables (< partition_size_mb), use single partition
        if estimated_size_mb < self._partition_size_mb:
            return {
                "use_partitioning": False,
                "num_partitions": 1,
                "reason": f"Small table ({estimated_size_mb:.2f} MB)"
            }
        
        # Calculate optimal number of partitions
        num_partitions = max(2, min(200, int(estimated_size_mb / self._partition_size_mb)))
        
        # Get partition column bounds if available
        if table_config.partition_column:
            lower_bound, upper_bound = self._metadata_provider.get_partition_bounds(table_config)
            if lower_bound is not None and upper_bound is not None:
                return {
                    "use_partitioning": True,
                    "partition_column": table_config.partition_column,
                    "lower_bound": int(lower_bound),
                    "upper_bound": int(upper_bound),
                    "num_partitions": num_partitions,
                    "reason": f"Large table ({estimated_size_mb:.2f} MB), using {num_partitions} partitions"
                }
        
        return {
            "use_partitioning": False,
            "num_partitions": num_partitions,
            "reason": f"No suitable partition column, will repartition after read"
        }


class FixedPartitionStrategy(PartitionStrategy):
    """Fixed partitioning strategy for consistent behavior."""
    
    def __init__(self, num_partitions: int = 10):
        self._num_partitions = num_partitions
    
    def calculate_partitions(self, table_config: TableConfig, row_count: int) -> Dict[str, Any]:
        if table_config.partition_column and table_config.lower_bound and table_config.upper_bound:
            return {
                "use_partitioning": True,
                "partition_column": table_config.partition_column,
                "lower_bound": table_config.lower_bound,
                "upper_bound": table_config.upper_bound,
                "num_partitions": table_config.num_partitions or self._num_partitions,
                "reason": "Fixed partitioning with provided bounds"
            }
        return {
            "use_partitioning": False,
            "num_partitions": self._num_partitions,
            "reason": "Fixed partitioning without bounds"
        }

## Cell 8: JDBC Data Reader (Single Responsibility)

In [ ]:
class JDBCDataReader:
    """Reads data from JDBC source - Single Responsibility: data reading only."""
    
    def __init__(self, 
                 spark: SparkSession,
                 connection_provider: IConnectionProvider,
                 partition_strategy: PartitionStrategy,
                 metadata_provider: ITableMetadataProvider):
        self._spark = spark
        self._conn_provider = connection_provider
        self._partition_strategy = partition_strategy
        self._metadata_provider = metadata_provider
    
    def read(self, table_config: TableConfig) -> DataFrame:
        """Read table data with optimal partitioning."""
        start_time = datetime.now()
        
        # Get row count for partitioning decisions
        row_count = self._metadata_provider.get_row_count(table_config)
        logger.info(f"Table {table_config.full_source_name}: {row_count:,} rows")
        
        # Calculate partition strategy
        partition_params = self._partition_strategy.calculate_partitions(table_config, row_count)
        logger.info(f"Partition strategy: {partition_params['reason']}")
        
        # Build the table/query reference
        if table_config.custom_query:
            table_ref = f"({table_config.custom_query}) as custom_query"
        else:
            table_ref = table_config.full_source_name
        
        # Read with or without partitioning
        if partition_params.get("use_partitioning"):
            df = self._spark.read.jdbc(
                url=self._conn_provider.get_jdbc_url(),
                table=table_ref,
                column=partition_params["partition_column"],
                lowerBound=partition_params["lower_bound"],
                upperBound=partition_params["upper_bound"],
                numPartitions=partition_params["num_partitions"],
                properties=self._conn_provider.get_connection_properties()
            )
        else:
            df = self._spark.read.jdbc(
                url=self._conn_provider.get_jdbc_url(),
                table=table_ref,
                properties=self._conn_provider.get_connection_properties()
            )
            # Repartition after read for better parallelism in downstream operations
            if partition_params["num_partitions"] > 1:
                df = df.repartition(partition_params["num_partitions"])
        
        # Add audit columns
        df = df.withColumn("_load_timestamp", current_timestamp())
        df = df.withColumn("_source_table", lit(table_config.full_source_name))
        
        elapsed = (datetime.now() - start_time).total_seconds()
        logger.info(f"Read completed in {elapsed:.2f}s")
        
        return df

## Cell 9: Delta Data Writer (Single Responsibility & Open/Closed)

In [ ]:
class DeltaDataWriter:
    """Writes data to Delta Lake - Single Responsibility: data writing only."""
    
    def __init__(self, spark: SparkSession, optimize_after_write: bool = True):
        self._spark = spark
        self._optimize_after_write = optimize_after_write
    
    def write(self, df: DataFrame, table_config: TableConfig, mode: str) -> Dict[str, Any]:
        """Write DataFrame to Delta table."""
        start_time = datetime.now()
        target_table = table_config.full_target_name
        
        if mode == "merge" and table_config.primary_keys:
            result = self._merge_write(df, table_config)
        elif mode == "append":
            result = self._append_write(df, target_table)
        else:  # overwrite
            result = self._overwrite_write(df, target_table)
        
        elapsed = (datetime.now() - start_time).total_seconds()
        result["write_time_seconds"] = elapsed
        
        # Cost optimization: run OPTIMIZE for better file layout
        if self._optimize_after_write:
            self._optimize_table(target_table)
        
        return result
    
    def _overwrite_write(self, df: DataFrame, target_table: str) -> Dict[str, Any]:
        """Full overwrite of target table."""
        df.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable(target_table)
        
        return {"mode": "overwrite", "status": "success"}
    
    def _append_write(self, df: DataFrame, target_table: str) -> Dict[str, Any]:
        """Append to target table."""
        df.write.format("delta").mode("append").saveAsTable(target_table)
        return {"mode": "append", "status": "success"}
    
    def _merge_write(self, df: DataFrame, table_config: TableConfig) -> Dict[str, Any]:
        """Merge into target table using primary keys."""
        target_table = table_config.full_target_name
        
        # Check if target exists
        if not self._spark.catalog.tableExists(target_table):
            # First load - do overwrite
            return self._overwrite_write(df, target_table)
        
        # Build merge condition
        delta_table = DeltaTable.forName(self._spark, target_table)
        merge_condition = " AND ".join(
            [f"target.{pk} = source.{pk}" for pk in table_config.primary_keys]
        )
        
        # Execute merge
        delta_table.alias("target").merge(
            df.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        
        return {"mode": "merge", "status": "success", "merge_keys": table_config.primary_keys}
    
    def _optimize_table(self, target_table: str) -> None:
        """Run OPTIMIZE to compact small files - Cost optimization."""
        try:
            self._spark.sql(f"OPTIMIZE {target_table}")
            logger.info(f"Optimized table {target_table}")
        except Exception as e:
            logger.warning(f"Optimize failed for {target_table}: {e}")

## Cell 10: Table Loader (Combines Reader & Writer - Dependency Inversion)

In [ ]:
class TableLoader:
    """Orchestrates table loading - Dependency Inversion: depends on abstractions."""
    
    def __init__(self, 
                 reader: IDataReader, 
                 writer: IDataWriter,
                 load_mode: str = "overwrite"):
        self._reader = reader
        self._writer = writer
        self._load_mode = load_mode
    
    def load(self, table_config: TableConfig) -> Dict[str, Any]:
        """Load a single table from source to target."""
        start_time = datetime.now()
        result = {
            "source_table": table_config.full_source_name,
            "target_table": table_config.full_target_name,
            "start_time": start_time.isoformat(),
            "status": "pending"
        }
        
        try:
            # Read from source
            logger.info(f"Loading table: {table_config.full_source_name}")
            df = self._reader.read(table_config)
            
            # Get row count before write
            row_count = df.count()
            result["row_count"] = row_count
            
            # Write to target
            write_result = self._writer.write(df, table_config, self._load_mode)
            result.update(write_result)
            
            result["status"] = "success"
            logger.info(f"Successfully loaded {row_count:,} rows to {table_config.full_target_name}")
            
        except Exception as e:
            result["status"] = "failed"
            result["error"] = str(e)
            logger.error(f"Failed to load {table_config.full_source_name}: {e}")
        
        finally:
            end_time = datetime.now()
            result["end_time"] = end_time.isoformat()
            result["duration_seconds"] = (end_time - start_time).total_seconds()
        
        return result

## Cell 11: Parallel Orchestrator (Cost Efficient Parallelism)

In [ ]:
class ParallelTableOrchestrator:
    """Orchestrates parallel table loading - Cost efficient with controlled parallelism."""
    
    def __init__(self, 
                 loader: ITableLoader,
                 max_parallel: int = 10):
        self._loader = loader
        self._max_parallel = max_parallel
    
    def load_tables(self, table_configs: List[TableConfig]) -> List[Dict[str, Any]]:
        """Load multiple tables in parallel with controlled concurrency."""
        results = []
        total_tables = len(table_configs)
        
        logger.info(f"Starting parallel load of {total_tables} tables with max {self._max_parallel} concurrent")
        start_time = datetime.now()
        
        # Use ThreadPoolExecutor for parallel execution
        # Spark operations are I/O bound so threading works well
        with ThreadPoolExecutor(max_workers=self._max_parallel) as executor:
            # Submit all tasks
            future_to_table = {
                executor.submit(self._loader.load, config): config 
                for config in table_configs
            }
            
            # Process results as they complete
            completed = 0
            for future in as_completed(future_to_table):
                table_config = future_to_table[future]
                try:
                    result = future.result()
                    results.append(result)
                except Exception as e:
                    results.append({
                        "source_table": table_config.full_source_name,
                        "target_table": table_config.full_target_name,
                        "status": "failed",
                        "error": str(e)
                    })
                
                completed += 1
                logger.info(f"Progress: {completed}/{total_tables} tables completed")
        
        total_duration = (datetime.now() - start_time).total_seconds()
        
        # Generate summary
        successful = sum(1 for r in results if r["status"] == "success")
        failed = sum(1 for r in results if r["status"] == "failed")
        total_rows = sum(r.get("row_count", 0) for r in results)
        
        logger.info(f"\n{'='*60}")
        logger.info(f"LOAD SUMMARY")
        logger.info(f"{'='*60}")
        logger.info(f"Total tables: {total_tables}")
        logger.info(f"Successful: {successful}")
        logger.info(f"Failed: {failed}")
        logger.info(f"Total rows loaded: {total_rows:,}")
        logger.info(f"Total duration: {total_duration:.2f}s")
        logger.info(f"{'='*60}")
        
        return results
    
    def load_tables_in_batches(self, 
                                table_configs: List[TableConfig],
                                batch_size: int = 20) -> List[Dict[str, Any]]:
        """Load tables in batches for very large table lists - Cost efficient memory management."""
        all_results = []
        total_batches = (len(table_configs) + batch_size - 1) // batch_size
        
        for i in range(0, len(table_configs), batch_size):
            batch_num = (i // batch_size) + 1
            batch = table_configs[i:i + batch_size]
            logger.info(f"\nProcessing batch {batch_num}/{total_batches} ({len(batch)} tables)")
            
            batch_results = self.load_tables(batch)
            all_results.extend(batch_results)
            
            # Clear any cached data between batches to free memory
            spark.catalog.clearCache()
        
        return all_results

## Cell 12: Table Discovery (Auto-discover tables from MSSQL)

In [ ]:
class TableDiscovery:
    """Discovers tables from source database - Single Responsibility."""
    
    def __init__(self, spark: SparkSession, connection_provider: IConnectionProvider):
        self._spark = spark
        self._conn_provider = connection_provider
    
    def discover_tables(self, 
                       schema: str,
                       target_catalog: str,
                       target_schema: str,
                       include_pattern: str = "%",
                       exclude_tables: List[str] = None) -> List[TableConfig]:
        """Discover all tables in a schema."""
        exclude_tables = exclude_tables or []
        
        query = f"""
            (SELECT 
                t.name as table_name,
                s.name as schema_name,
                p.rows as row_count,
                STUFF((
                    SELECT ',' + c.name
                    FROM sys.index_columns ic
                    JOIN sys.columns c ON ic.object_id = c.object_id AND ic.column_id = c.column_id
                    JOIN sys.indexes i ON ic.object_id = i.object_id AND ic.index_id = i.index_id
                    WHERE i.is_primary_key = 1 AND i.object_id = t.object_id
                    FOR XML PATH('')
                ), 1, 1, '') as primary_keys,
                (
                    SELECT TOP 1 c.name
                    FROM sys.columns c
                    JOIN sys.types tp ON c.user_type_id = tp.user_type_id
                    WHERE c.object_id = t.object_id
                    AND tp.name IN ('int', 'bigint')
                    AND c.is_identity = 1
                ) as identity_column
             FROM sys.tables t
             JOIN sys.schemas s ON t.schema_id = s.schema_id
             JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0, 1)
             WHERE s.name = '{schema}'
               AND t.name LIKE '{include_pattern}'
               AND t.is_ms_shipped = 0
             GROUP BY t.object_id, t.name, s.name, p.rows) as tables_query
        """
        
        df = self._spark.read.jdbc(
            url=self._conn_provider.get_jdbc_url(),
            table=query,
            properties=self._conn_provider.get_connection_properties()
        )
        
        table_configs = []
        for row in df.collect():
            if row.table_name in exclude_tables:
                continue
            
            pk_list = row.primary_keys.split(',') if row.primary_keys else []
            
            config = TableConfig(
                source_schema=row.schema_name,
                source_table=row.table_name,
                target_catalog=target_catalog,
                target_schema=target_schema,
                partition_column=row.identity_column,
                primary_keys=pk_list
            )
            table_configs.append(config)
        
        logger.info(f"Discovered {len(table_configs)} tables in schema {schema}")
        return table_configs

## Cell 13: Load Results Reporter

In [ ]:
class LoadResultsReporter:
    """Reports and persists load results - Single Responsibility."""
    
    def __init__(self, spark: SparkSession):
        self._spark = spark
    
    def create_report_dataframe(self, results: List[Dict[str, Any]]) -> DataFrame:
        """Convert results to a DataFrame for analysis."""
        return self._spark.createDataFrame(results)
    
    def save_results(self, results: List[Dict[str, Any]], 
                     catalog: str, schema: str, table: str = "_load_audit_log") -> None:
        """Persist results to audit table."""
        df = self.create_report_dataframe(results)
        df.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.{table}")
        logger.info(f"Saved {len(results)} load results to {catalog}.{schema}.{table}")
    
    def display_summary(self, results: List[Dict[str, Any]]) -> None:
        """Display summary statistics."""
        df = self.create_report_dataframe(results)
        
        print("\n" + "="*80)
        print("LOAD EXECUTION SUMMARY")
        print("="*80)
        
        # Status breakdown
        print("\nStatus Breakdown:")
        df.groupBy("status").count().show()
        
        # Top 10 longest running
        print("\nTop 10 Longest Running Tables:")
        df.filter(col("status") == "success") \
          .orderBy(col("duration_seconds").desc()) \
          .select("source_table", "row_count", "duration_seconds") \
          .limit(10) \
          .show(truncate=False)
        
        # Failed tables
        failed = df.filter(col("status") == "failed")
        if failed.count() > 0:
            print("\nFailed Tables:")
            failed.select("source_table", "error").show(truncate=False)
        
        print("="*80)

## Cell 14: Factory for Creating Components (Dependency Injection)

In [ ]:
class LoaderFactory:
    """Factory for creating loader components - Dependency Inversion."""
    
    @staticmethod
    def create_loader(spark: SparkSession, config: LoaderConfig) -> ParallelTableOrchestrator:
        """Create a fully configured parallel loader."""
        
        # Create connection provider
        conn_provider = MSSQLConnectionProvider(config.jdbc_config)
        
        # Create metadata provider
        metadata_provider = MSSQLMetadataProvider(spark, conn_provider)
        
        # Create partition strategy
        partition_strategy = AdaptivePartitionStrategy(
            metadata_provider=metadata_provider,
            partition_size_mb=config.partition_size_mb
        )
        
        # Create reader
        reader = JDBCDataReader(
            spark=spark,
            connection_provider=conn_provider,
            partition_strategy=partition_strategy,
            metadata_provider=metadata_provider
        )
        
        # Create writer
        writer = DeltaDataWriter(spark)
        
        # Create table loader
        table_loader = TableLoader(reader, writer, config.load_mode)
        
        # Create orchestrator
        orchestrator = ParallelTableOrchestrator(
            loader=table_loader,
            max_parallel=config.max_parallel_tables
        )
        
        return orchestrator
    
    @staticmethod
    def create_table_discovery(spark: SparkSession, config: LoaderConfig) -> TableDiscovery:
        """Create a table discovery instance."""
        conn_provider = MSSQLConnectionProvider(config.jdbc_config)
        return TableDiscovery(spark, conn_provider)

## Cell 15: Main Execution - Initialize Configuration

In [ ]:
# Get widget values
mssql_host = dbutils.widgets.get("mssql_host")
mssql_port = int(dbutils.widgets.get("mssql_port"))
mssql_database = dbutils.widgets.get("mssql_database")
mssql_schema = dbutils.widgets.get("mssql_schema")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
load_mode = dbutils.widgets.get("load_mode")
max_parallel_tables = int(dbutils.widgets.get("max_parallel_tables"))
partition_size_mb = int(dbutils.widgets.get("partition_size_mb"))

# Create JDBC configuration
jdbc_config = JDBCConfig(
    host=mssql_host,
    port=mssql_port,
    database=mssql_database
)

# Create loader configuration
loader_config = LoaderConfig(
    jdbc_config=jdbc_config,
    max_parallel_tables=max_parallel_tables,
    partition_size_mb=partition_size_mb,
    load_mode=load_mode
)

print(f"Configuration initialized:")
print(f"  Source: {mssql_host}/{mssql_database}.{mssql_schema}")
print(f"  Target: {target_catalog}.{target_schema}")
print(f"  Load Mode: {load_mode}")
print(f"  Max Parallel Tables: {max_parallel_tables}")
print(f"  Partition Size: {partition_size_mb} MB")

## Cell 16: Discover Tables to Load

In [ ]:
# Create table discovery
discovery = LoaderFactory.create_table_discovery(spark, loader_config)

# Discover all tables in the schema
# Optionally exclude specific tables
exclude_tables = [
    # Add tables to exclude here
    # "sysdiagrams",
    # "__EFMigrationsHistory",
]

table_configs = discovery.discover_tables(
    schema=mssql_schema,
    target_catalog=target_catalog,
    target_schema=target_schema,
    exclude_tables=exclude_tables
)

print(f"\nDiscovered {len(table_configs)} tables to load:")
for i, config in enumerate(table_configs[:10], 1):
    print(f"  {i}. {config.full_source_name} -> {config.full_target_name}")
if len(table_configs) > 10:
    print(f"  ... and {len(table_configs) - 10} more tables")

## Cell 17: Alternative - Manual Table List (if not using discovery)

In [ ]:
# Alternatively, define tables manually if you need specific configurations
# Uncomment and modify as needed

# manual_table_configs = [
#     TableConfig(
#         source_schema="dbo",
#         source_table="Customers",
#         target_catalog=target_catalog,
#         target_schema=target_schema,
#         partition_column="CustomerID",  # Specify partition column for large tables
#         primary_keys=["CustomerID"]
#     ),
#     TableConfig(
#         source_schema="dbo",
#         source_table="Orders",
#         target_catalog=target_catalog,
#         target_schema=target_schema,
#         partition_column="OrderID",
#         primary_keys=["OrderID"]
#     ),
#     TableConfig(
#         source_schema="dbo",
#         source_table="OrderDetails",
#         target_catalog=target_catalog,
#         target_schema=target_schema,
#         partition_column="OrderDetailID",
#         primary_keys=["OrderDetailID"]
#     ),
#     # Add more tables as needed...
# ]

# # Use manual configs instead of discovered ones
# table_configs = manual_table_configs

## Cell 18: Create Target Schema if Not Exists

In [ ]:
# Ensure target catalog and schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

print(f"Target schema {target_catalog}.{target_schema} is ready")

## Cell 19: Execute Parallel Load

In [ ]:
# Create the parallel loader
orchestrator = LoaderFactory.create_loader(spark, loader_config)

# Execute parallel load
# For very large table lists (100+), use batched loading
if len(table_configs) > 50:
    # Load in batches to manage memory and provide checkpoints
    results = orchestrator.load_tables_in_batches(
        table_configs=table_configs,
        batch_size=25  # Process 25 tables per batch
    )
else:
    # Load all tables in parallel
    results = orchestrator.load_tables(table_configs)

## Cell 20: Generate and Save Report

In [ ]:
# Create reporter and display results
reporter = LoadResultsReporter(spark)

# Display summary
reporter.display_summary(results)

# Save results to audit table
reporter.save_results(
    results=results,
    catalog=target_catalog,
    schema=target_schema,
    table="_load_audit_log"
)

## Cell 21: View Failed Tables (if any)

In [ ]:
# Filter and display failed tables for retry
failed_tables = [r for r in results if r["status"] == "failed"]

if failed_tables:
    print(f"\n{len(failed_tables)} tables failed to load:\n")
    for ft in failed_tables:
        print(f"Table: {ft['source_table']}")
        print(f"Error: {ft.get('error', 'Unknown error')}")
        print("-" * 40)
    
    # Create configs for retry
    failed_table_names = [ft["source_table"] for ft in failed_tables]
    retry_configs = [tc for tc in table_configs if tc.full_source_name in failed_table_names]
    print(f"\n{len(retry_configs)} table configs ready for retry")
else:
    print("All tables loaded successfully!")

## Cell 22: Retry Failed Tables (Optional)

In [ ]:
# Uncomment to retry failed tables with potentially different settings

# if retry_configs:
#     # Create a new loader with different settings for retry
#     retry_loader_config = LoaderConfig(
#         jdbc_config=jdbc_config,
#         max_parallel_tables=5,  # Reduced parallelism
#         partition_size_mb=64,   # Smaller partitions
#         load_mode=load_mode
#     )
#     
#     retry_orchestrator = LoaderFactory.create_loader(spark, retry_loader_config)
#     retry_results = retry_orchestrator.load_tables(retry_configs)
#     
#     reporter.display_summary(retry_results)

## Cell 23: Final Validation

In [ ]:
# Validate loaded tables
print("Validating loaded tables...\n")

validation_results = []
successful_results = [r for r in results if r["status"] == "success"]

for result in successful_results[:10]:  # Validate first 10
    target_table = result["target_table"]
    try:
        count = spark.table(target_table).count()
        expected = result.get("row_count", 0)
        match = "✓" if count == expected else "✗"
        validation_results.append({
            "table": target_table,
            "expected": expected,
            "actual": count,
            "match": match
        })
    except Exception as e:
        validation_results.append({
            "table": target_table,
            "error": str(e)
        })

# Display validation results
validation_df = spark.createDataFrame(validation_results)
validation_df.show(truncate=False)

print(f"\nValidation complete for {len(validation_results)} tables")

## Cost Optimization Tips

1. **Cluster Sizing**: Use auto-scaling clusters with min/max workers based on table count
2. **Partition Size**: Adjust `partition_size_mb` based on your data - larger = fewer partitions = less overhead
3. **Parallel Tables**: Balance `max_parallel_tables` with cluster size - too many can cause resource contention
4. **Batch Loading**: For 100+ tables, use `load_tables_in_batches()` to free memory between batches
5. **Query Pushdown**: Enabled by default - reduces data transfer from MSSQL
6. **Delta OPTIMIZE**: Runs after each table - reduces small files and improves query performance
7. **Photon**: Enable Photon runtime for faster Delta Lake operations

## Recommended Cluster Configuration

```
- Node type: Standard_DS3_v2 or similar
- Min workers: 2
- Max workers: 10 (for 100 tables)
- Autoscaling: Enabled
- Photon: Enabled
- Spark config:
    spark.sql.shuffle.partitions 200
    spark.databricks.delta.optimizeWrite.enabled true
    spark.databricks.delta.autoCompact.enabled true
```